# Visualize attention pattern in few-shot prompts

## Loading packages

In [ ]:
import circuitsvis
import circuitsvis.attention as cv_attn
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm import tqdm
from typing import List, Dict, Set
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching
except ImportError:
    print("Installing transformer-lens for mechanistic interpretability...")
    !pip install transformer-lens -q
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load utils
REPO_ROOT = Path(r"C:\Users\nguye\repro-shared-lexical-task")
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils.build_prompts import create_few_shot_prompts

# Global variables
DATASET_FOLDER = str(REPO_ROOT / "datasets" / "abstractive")

Using device: cpu


In [ ]:
# from huggingface_hub import login
# login()

# Llama-3.1-8B-Instruct
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map="auto")

# Llama-3.2-1B-Instruct
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", device_map="auto")

model = HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

C:\Users\nguye\AppData\Local\Temp\ipykernel_29752\3652073285.py:12: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
Loading weights: 100%|██████████| 146/146 [00:04<00:00, 33.08it/s]


Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer


## Loading prompts

In [8]:
prompts, answers, _ = create_few_shot_prompts(
    d_name="country-capital",
    n_shot=5,
    dataset_folder=DATASET_FOLDER, 
    delimiter=";", q_bos=" ", a_bos=" ", qa_delimiter=":")

In [10]:
prompt = prompts[5] 

## Attention patterns

In [ ]:
MODEL_SHORT = "Llama-3.2-1B-Instruct"
D_NAME = "country-capital"
PATTERN_FOLDER = REPO_ROOT / "output" / MODEL_SHORT / D_NAME / "Heads" / "chunking" / "attn_pattern"
PATTERN_FOLDER.mkdir(parents=True, exist_ok=True)

In [17]:
prompt

' Afghanistan: Kabul; Albania: Tirana; Algeria: Algiers; Andorra: Andorra la Vella; Angola: Luanda; Antigua and Barbuda:'

In [16]:
str_tokens = model.to_str_tokens(prompt)   # includes the prepended BOS automatically
print(str_tokens, len(str_tokens))

['<|begin_of_text|>', ' Afghanistan', ':', ' Kabul', ';', ' Albania', ':', ' Tir', 'ana', ';', ' Algeria', ':', ' Alg', 'iers', ';', ' And', 'orra', ':', ' And', 'orra', ' la', ' V', 'ella', ';', ' Angola', ':', ' Lu', 'anda', ';', ' Ant', 'igua', ' and', ' Barb', 'uda', ':'] 35


In [ ]:
def visualize_attn_pattern(prompt, layer, head):
    tokens = model.to_tolens(prompt)
    _, cache = model.run_with_cache(
        tokens, names_filter=lambda name: name == f"blocls.{layer}.attn.hook_pattern"
    )
    pattern = cache[f"blocks.{layer}.attn.hook_pattern"] # [1, n_heads, seq, seq]
    head_pattern = pattern[0, head] # [seq, seq] -- rows=query pos, cols=key pos

    fig = px.imshow(
        head_pattern.cpu().numpy(),
        x=str_tokens, y=str_tokens,
        labels={"x": "key position", "y": "query position", "color": "attention weight"},
        title=f"L{layer}H{head} attention pattern",
        color_continuous_scale="Greens",
    )

    # fig.write_html(PATTERN_FOLDER / f"L{layer}H{head}_attn_pattern.html")

    vis = cv_attn.attention_patterns(
        tokens=str_tokens,
        attention=pattern[0, head].unsqueeze(0),   # [1, seq, seq]
    )

for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))


In [35]:
tokens = model.to_tokens(prompt)  # [1, seq]
layer = 6
_, cache = model.run_with_cache(
    tokens, names_filter=lambda name: name == f"blocks.{layer}.attn.hook_pattern"
)
pattern = cache[f"blocks.{layer}.attn.hook_pattern"]   # [1, n_heads, seq, seq]

In [36]:
head = 6
head_pattern = pattern[0, head]   # [seq, seq] -- rows=query pos, cols=key pos

In [37]:
MODEL_SHORT = "Llama-3.2-1B-Instruct"
D_NAME = "country-capital"
PATTERN_FOLDER = REPO_ROOT / "output" / MODEL_SHORT / D_NAME / "Heads" / "chunking" / "attn_pattern"
PATTERN_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:

fig = px.imshow(
    head_pattern.cpu().numpy(),
    x=str_tokens, y=str_tokens,
    labels={"x": "key position", "y": "query position", "color": "attention weight"},
    title=f"L{layer}H{head} attention pattern",
    color_continuous_scale="Greens",
)

fig.write_html(PATTERN_FOLDER / f"L{layer}H{head}_attn_pattern.html")

In [39]:
# pattern: [1, n_heads, seq, seq]
# all heads in the layer, with a head-selector dropdown
vis = cv_attn.attention_patterns(tokens=str_tokens, attention=pattern[0])
vis = cv_attn.attention_patterns(
    tokens=str_tokens,
    attention=pattern[0, head].unsqueeze(0),   # [1, seq, seq]
)
vis